# Player-Level EDA: Does Club Strength Affect World Cup Performance?

**This notebook answers three questions before we build squad-level features:**

1. Does a player's club strength (Club Elo) correlate with their World Cup performance?
2. Do players from stronger clubs score more at the World Cup?
3. Does a player's club form (goals, assists, minutes) predict their World Cup contribution?

**Key improvement over hardcoded tiers:**  
Club strength is measured using **Club Elo ratings** from [clubelo.com](http://clubelo.com) — a continuous, data-driven score based on actual match results.

**Data:** `player_stats_with_club_elo.csv` — player stats enriched with Club Elo ratings  
**Coverage:** 63.7% direct club match, 20.1% league average fallback, 16.3% no match

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = "../../data"
PLAYER_PATH = f"{DATA_DIR}/processed/player_stats_with_club_elo.csv"

df = pd.read_csv(PLAYER_PATH)

# Keep only players with club data found
df_found = df[df["perf_found"] == True].copy()

# Players with club strength score
df_elo = df_found[df_found["club_strength"].notna()].copy()

print(f"Total players:                  {len(df)}")
print(f"With club data:                 {len(df_found)} ({len(df_found)/len(df):.1%})")
print(
    f"With club strength (Elo):       {len(df_elo)} ({len(df_elo)/len(df_found):.1%} of those with club data)"
)
print(
    f"\nClub strength range: {df_elo['club_strength'].min():.0f} – {df_elo['club_strength'].max():.0f}"
)
print(f"Median club strength:           {df_elo['club_strength'].median():.0f}")

## 1. Club Strength Distribution

Before asking whether club strength predicts WC performance, we understand what the distribution looks like.

In [ ]:
# Define strength bands based on Elo percentiles
# Top 25% = Elite (>1847), 50-75% = Strong (1723-1847), 25-50% = Mid (1631-1723), Bottom 25% = Weak (<1631)
q25, q50, q75 = df_elo["club_strength"].quantile([0.25, 0.50, 0.75])


def strength_band(elo):
    if elo >= q75:
        return "Elite (top 25%)"
    if elo >= q50:
        return "Strong (50-75%)"
    if elo >= q25:
        return "Mid (25-50%)"
    return "Weak (bottom 25%)"


df_elo["strength_band"] = df_elo["club_strength"].apply(strength_band)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(
    "Club Strength Distribution (Club Elo Ratings, 2006–2022)",
    fontsize=13,
    fontweight="bold",
)

# Histogram
axes[0].hist(df_elo["club_strength"], bins=40, color="#3498db", edgecolor="white")
for q, label in [(q25, "Q25"), (q50, "Median"), (q75, "Q75")]:
    axes[0].axvline(q, color="red", linestyle="--", alpha=0.7)
    axes[0].text(q + 5, axes[0].get_ylim()[1] * 0.9, label, fontsize=8, color="red")
axes[0].set_xlabel("Club Elo Rating")
axes[0].set_ylabel("Number of Players")
axes[0].set_title("Distribution of Club Strength")

# Band breakdown
band_order = ["Elite (top 25%)", "Strong (50-75%)", "Mid (25-50%)", "Weak (bottom 25%)"]
band_counts = df_elo["strength_band"].value_counts().reindex(band_order)
colors = ["#e74c3c", "#e67e22", "#3498db", "#95a5a6"]
axes[1].bar(band_counts.index, band_counts.values, color=colors)
axes[1].set_title("Players by Strength Band")
axes[1].set_ylabel("Number of Players")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print(f"\nStrength band thresholds:")
print(f"  Elite  (top 25%):    Elo ≥ {q75:.0f}")
print(f"  Strong (50-75%):     Elo {q50:.0f} – {q75:.0f}")
print(f"  Mid    (25-50%):     Elo {q25:.0f} – {q50:.0f}")
print(f"  Weak   (bottom 25%): Elo < {q25:.0f}")

## Question 1: Does Club Strength Correlate with WC Performance?

In [ ]:
band_order = ["Elite (top 25%)", "Strong (50-75%)", "Mid (25-50%)", "Weak (bottom 25%)"]
colors = ["#e74c3c", "#e67e22", "#3498db", "#95a5a6"]

band_stats = (
    df_elo.groupby("strength_band")
    .agg(
        n_players=("player_id", "count"),
        avg_wc_goals=("wc_goals", "mean"),
        pct_scored=("wc_goals", lambda x: (x > 0).mean()),
        avg_matches=("matches_played", "mean"),
        avg_starts=("starts", "mean"),
    )
    .reindex(band_order)
    .round(3)
)

print("WC performance by club strength band:\n")
print(band_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle(
    "World Cup Performance by Club Strength Band (2006–2022)",
    fontsize=13,
    fontweight="bold",
)

for ax, (col, title) in zip(
    axes,
    [
        ("avg_wc_goals", "Avg WC Goals per Player"),
        ("pct_scored", "% of Players Who Scored"),
        ("avg_matches", "Avg WC Matches Played"),
    ],
):
    vals = band_stats[col]
    bars = ax.bar(range(len(band_order)), vals, color=colors)
    ax.set_xticks(range(len(band_order)))
    ax.set_xticklabels([b.split(" (")[0] for b in band_order], rotation=15, ha="right")
    ax.set_title(title, fontsize=11)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()

# Statistical test: Elite vs Weak
elite = df_elo[df_elo["strength_band"] == "Elite (top 25%)"]["wc_goals"]
weak = df_elo[df_elo["strength_band"] == "Weak (bottom 25%)"]["wc_goals"]
_, p = stats.mannwhitneyu(elite, weak, alternative="greater")
print(f"\nMann-Whitney U test (Elite WC goals > Weak):")
print(f"  p-value = {p:.4f} {'→ Significant ✓' if p < 0.05 else '→ Not significant'}")

## Question 2: Scatter — Club Elo vs WC Goals (by Position)

In [ ]:
# Spearman correlation: club_strength → wc_goals by position
print("Spearman correlation: club_strength → wc_goals by position:\n")
for pos in ["forward", "midfielder", "defender", "goal keeper"]:
    sub = df_elo[df_elo["position_name"] == pos]
    rho, p = stats.spearmanr(sub["club_strength"], sub["wc_goals"])
    print(
        f"  {pos:<15}  n={len(sub):>4}   r={rho:.3f}   p={p:.4f}  {'✓' if p < 0.05 else ''}"
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Club Strength vs WC Goals", fontsize=13, fontweight="bold")

# All players
axes[0].scatter(
    df_elo["club_strength"], df_elo["wc_goals"], alpha=0.3, s=15, color="#3498db"
)
m, b = np.polyfit(df_elo["club_strength"], df_elo["wc_goals"], 1)
x = np.linspace(df_elo["club_strength"].min(), df_elo["club_strength"].max(), 100)
axes[0].plot(x, m * x + b, color="red", linewidth=2)
rho, p = stats.spearmanr(df_elo["club_strength"], df_elo["wc_goals"])
axes[0].set_xlabel("Club Elo Rating")
axes[0].set_ylabel("WC Goals")
axes[0].set_title(f"All positions  (r={rho:.3f}, p={p:.4f})")

# Forwards only
fwd = df_elo[df_elo["position_name"] == "forward"]
axes[1].scatter(fwd["club_strength"], fwd["wc_goals"], alpha=0.4, s=20, color="#e74c3c")
m2, b2 = np.polyfit(fwd["club_strength"], fwd["wc_goals"], 1)
axes[1].plot(x, m2 * x + b2, color="darkred", linewidth=2)
rho2, p2 = stats.spearmanr(fwd["club_strength"], fwd["wc_goals"])
axes[1].set_xlabel("Club Elo Rating")
axes[1].set_ylabel("WC Goals")
axes[1].set_title(f"Forwards only  (r={rho2:.3f}, p={p2:.4f})")

plt.tight_layout()
plt.show()

## Question 3: Does Club Form Predict WC Contribution?

In [ ]:
club_form_cols = [
    "club_strength",
    "minutes_played",
    "appearances",
    "club_goals",
    "assists",
]
wc_perf_cols = ["wc_goals", "matches_played", "starts"]

print("Spearman correlation: club form + strength → WC performance\n")
corr_rows = []
for wc_col in wc_perf_cols:
    for club_col in club_form_cols:
        sub = df_elo[[club_col, wc_col]].dropna()
        rho, p = stats.spearmanr(sub[club_col], sub[wc_col])
        corr_rows.append(
            {
                "Club metric": club_col,
                "WC metric": wc_col,
                "Spearman r": round(rho, 3),
                "p-value": round(p, 4),
                "Sig": "✓" if p < 0.05 else "",
            }
        )

corr_df = pd.DataFrame(corr_rows)
print(corr_df.to_string(index=False))

# Heatmap
pivot = corr_df.pivot(index="Club metric", columns="WC metric", values="Spearman r")
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    ax=ax,
    vmin=-0.3,
    vmax=0.3,
)
ax.set_title("Spearman Correlation: Club Metrics → WC Performance", fontsize=11)
plt.tight_layout()
plt.show()

## Summary of Findings

In [ ]:
elite_goals = df_elo[df_elo["strength_band"] == "Elite (top 25%)"]["wc_goals"].mean()
weak_goals = df_elo[df_elo["strength_band"] == "Weak (bottom 25%)"]["wc_goals"].mean()
rho_elo, p_elo = stats.spearmanr(df_elo["club_strength"], df_elo["wc_goals"])
rho_clg, p_clg = stats.spearmanr(
    df_elo["club_goals"].dropna(), df_elo["wc_goals"][df_elo["club_goals"].notna()]
)
rho_mins, p_mins = stats.spearmanr(
    df_elo["minutes_played"].dropna(),
    df_elo["matches_played"][df_elo["minutes_played"].notna()],
)

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(
    f"""
Q1 — Club strength (Elo) vs WC goals:
   Elite club avg WC goals : {elite_goals:.4f}
   Weak  club avg WC goals : {weak_goals:.4f}
   Elite players score {elite_goals/max(weak_goals,0.0001):.1f}x more on average
   Spearman r = {rho_elo:.3f} (p={p_elo:.4f}) {'✓ significant' if p_elo<0.05 else ''}

Q2 — Club goals → WC goals:
   Spearman r = {rho_clg:.3f} (p={p_clg:.4f}) {'✓ significant' if p_clg<0.05 else ''}

Q3 — Minutes played → WC matches:
   Spearman r = {rho_mins:.3f} (p={p_mins:.4f}) {'✓ significant' if p_mins<0.05 else ''}
"""
)
print("These findings justify using club_strength (Club Elo) and")
print("club form as squad-level features in 05_squad_level_features.ipynb")